## Inferencia sobre validacion — metricas CV (STL disease)

Calcula las metricas de los 5 folds de cada modelo sobre su conjunto de **validacion** retenido. Reconstruye el `val_idx` de cada fold con el mismo `StratifiedKFold(random_state=42)` del entrenamiento, carga los `best_weights` guardados y predice. 

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, roc_auc_score,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"   

train_csv  = "/home/marc/MARIADELMAR_EXPERIMENTS/onehot_files/train_onehot.csv"
val_csv    = "/home/marc/MARIADELMAR_EXPERIMENTS/onehot_files/val_onehot.csv"
images_dir = "/home/marc/MARIADELMAR_EXPERIMENTS/dataverse_files/images"

NUM_CLASSES = 7
N_FOLDS     = 5
class_cols  = ["dx_akiec", "dx_bcc", "dx_bkl", "dx_df", "dx_mel", "dx_nv", "dx_vasc"]
class_names = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

# Mismo conjunto train+val y mismo split que en los notebooks de entrenamiento
df_trainval = pd.concat([pd.read_csv(train_csv), pd.read_csv(val_csv)], ignore_index=True)
df_trainval["filepath"] = df_trainval["image_id"].apply(lambda x: os.path.join(images_dir, f"{x}.jpg"))
y_trainval_int = np.argmax(df_trainval[class_cols].values.astype("float32"), axis=1)

skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(skf.split(np.zeros(len(df_trainval)), y_trainval_int))


def make_ds(filepaths, img_size, preprocess_fn, batch=32):
    
    def _load(fp):
        img = tf.io.read_file(fp)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)
        return preprocess_fn(img)
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(_load, num_parallel_calls=15)
    return ds.batch(batch).prefetch(50)


def compute_all_metrics(y_true_int, proba):
    pred = np.argmax(proba, axis=1)
    m = {
        "disease_acc":             float((y_true_int == pred).mean()),
        "disease_balanced_acc":    float(balanced_accuracy_score(y_true_int, pred)),
        "disease_precision_macro": float(precision_score(y_true_int, pred, average="macro", zero_division=0)),
        "disease_recall_macro":    float(recall_score(y_true_int, pred, average="macro", zero_division=0)),
        "disease_f1_macro":        float(f1_score(y_true_int, pred, average="macro", zero_division=0)),
        "disease_f1_weighted":     float(f1_score(y_true_int, pred, average="weighted", zero_division=0)),
        "disease_kappa":           float(cohen_kappa_score(y_true_int, pred)),
    }
    try:
        m["disease_auc_macro"] = float(roc_auc_score(np.eye(NUM_CLASSES)[y_true_int],
                                       proba, multi_class="ovr", average="macro"))
    except Exception:
        m["disease_auc_macro"] = float("nan")
    m["auc_melanoma"] = float(roc_auc_score((y_true_int == 4).astype(int), proba[:, 4]))
    return m

2026-06-23 09:53:54.045782: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1


In [2]:
# Preprocess por arquitectura 
preprocess_vgg16       = lambda img: tf.keras.applications.vgg16.preprocess_input(img)
preprocess_resnet50    = lambda img: tf.keras.applications.resnet.preprocess_input(img)
preprocess_mobilenetv2 = lambda img: tf.keras.applications.mobilenet_v2.preprocess_input(img)
preprocess_densenet169 = lambda img: tf.keras.applications.densenet.preprocess_input(img)
preprocess_unet        = lambda img: img / 255.0

In [3]:
def build_vgg16(img_size):
    base = tf.keras.applications.VGG16(include_top=False, weights="imagenet",
                                       input_shape=(img_size[0], img_size[1], 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_disease")(x)
    return tf.keras.Model(inputs, out, name="STL_VGG16_disease")

def build_resnet50(img_size):
    base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                                          input_shape=(img_size[0], img_size[1], 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_disease")(x)
    return tf.keras.Model(inputs, out, name="STL_ResNet50_disease")

def build_mobilenetv2(img_size):
    base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                             input_shape=(img_size[0], img_size[1], 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_disease")(x)
    return tf.keras.Model(inputs, out, name="STL_MobileNetV2_disease")

def build_densenet169(img_size):
    base = tf.keras.applications.DenseNet169(include_top=False, weights="imagenet",
                                             input_shape=(img_size[0], img_size[1], 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_disease")(x)
    return tf.keras.Model(inputs, out, name="STL_DenseNet169_disease")

def build_unet(img_size):
    # U-Net autoencoder completo (encoder + decoder), 2 salidas: [reconstruccion, clasificacion].
    # Para el run nuevo de la U-Net. El bucle de inferencia ya coge proba[1] (la clasificacion).
    def encoder_block(inputs, num_filters):
        x = tf.keras.layers.Conv2D(num_filters, 3, activation="relu", padding="same",
                                   kernel_initializer="he_normal")(inputs)
        x = tf.keras.layers.Conv2D(num_filters, 3, activation="relu", padding="same",
                                   kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        p = tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(x)
        return x, p

    def decoder_block(inputs, skip_features, num_filters):
        up = tf.keras.layers.UpSampling2D(size=(2, 2))(inputs)
        x  = tf.keras.layers.Conv2D(num_filters, 2, activation="relu", padding="same",
                                    kernel_initializer="he_normal")(up)
        x = tf.keras.layers.concatenate([skip_features, x], axis=3)
        x = tf.keras.layers.Conv2D(num_filters, 3, activation="relu", padding="same",
                                   kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(num_filters, 3, activation="relu", padding="same",
                                   kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        return x

    inputs = tf.keras.layers.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    c1, p1 = encoder_block(inputs, 64)
    c2, p2 = encoder_block(p1, 128)
    c3, p3 = encoder_block(p2, 256)
    c4, p4 = encoder_block(p3, 512)
    b1 = tf.keras.layers.Conv2D(1024, 3, activation="relu", padding="same",
                                kernel_initializer="he_normal")(p4)
    b1 = tf.keras.layers.BatchNormalization()(b1)
    b1 = tf.keras.layers.Conv2D(1024, 3, activation="relu", padding="same",
                                kernel_initializer="he_normal", name="bottleneck_conv")(b1)
    b1 = tf.keras.layers.BatchNormalization(name="bottleneck_bn")(b1)
    e1 = decoder_block(b1, c4, 512)
    e2 = decoder_block(e1, c3, 256)
    e3 = decoder_block(e2, c2, 128)
    e4 = decoder_block(e3, c1, 64)
    final_recon = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same",
                                         kernel_initializer="he_normal")(e4)
    final_recon = tf.keras.layers.BatchNormalization()(final_recon)
    reconstruction_output = tf.keras.layers.Conv2D(3, 1, activation="relu", padding="same",
                                                   name="reconstruction_output")(final_recon)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(b1)
    shared = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    shared = tf.keras.layers.Dropout(0.3, name="shared_dropout")(shared)
    out_disease = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_disease")(shared)
    return tf.keras.Model(inputs, [reconstruction_output, out_disease],
                          name="STL_UNet_disease_autoencoder")

In [4]:
BASE_STL = Path("/home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos")

MODELS = [
    {
        "name":       "VGG16",
        "img_size":   (224, 224),
        "preprocess": preprocess_vgg16,
        "build_fn":   build_vgg16,
        "exp_dir":    BASE_STL / "VGG16_STL" / "exp_2026-04-28_18-45_disease_CUT7_5fold",
    },
    {
        "name":       "ResNet50",
        "img_size":   (224, 224),
        "preprocess": preprocess_resnet50,
        "build_fn":   build_resnet50,
        "exp_dir":    BASE_STL / "ResNet_STL" / "exp_2026-04-26_16-36_disease_CUT39_5fold",
    },
    {
        "name":       "MobileNetV2",
        "img_size":   (224, 224),
        "preprocess": preprocess_mobilenetv2,
        "build_fn":   build_mobilenetv2,
        "exp_dir":    BASE_STL / "MobileNetV2_STL" / "exp_2026-04-26_14-03_disease_CUT0_5fold",
    },
    {
        "name":       "DenseNet169",
        "img_size":   (224, 224),
        "preprocess": preprocess_densenet169,
        "build_fn":   build_densenet169,
        "exp_dir":    BASE_STL / "DenseNet169_STL" / "exp_2026-04-26_11-05_disease_CUT0_5fold",
    },
    
    {
         "name":       "UNet",
         "img_size":   (256, 256),
         "preprocess": preprocess_unet,
         "build_fn":   build_unet,
         "exp_dir":    BASE_STL / "UNet_STL" / "exp_2026-06-21_23-15_disease_5fold",
    },
]

# Verificacion de rutas
print("Verificando rutas de experimentos:")
for m in MODELS:
    ok = m["exp_dir"].exists()
    print(f"  [{'OK' if ok else 'FALTA'}] {m['name']:<14} {m['exp_dir']}")

Verificando rutas de experimentos:
  [OK] VGG16          /home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos/VGG16_STL/exp_2026-04-28_18-45_disease_CUT7_5fold
  [OK] ResNet50       /home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos/ResNet_STL/exp_2026-04-26_16-36_disease_CUT39_5fold
  [OK] MobileNetV2    /home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos/MobileNetV2_STL/exp_2026-04-26_14-03_disease_CUT0_5fold
  [OK] DenseNet169    /home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos/DenseNet169_STL/exp_2026-04-26_11-05_disease_CUT0_5fold
  [OK] UNet           /home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos/UNet_STL/exp_2026-06-21_23-15_disease_5fold


In [5]:
for cfg in MODELS:
    name    = cfg["name"]
    exp_dir = Path(cfg["exp_dir"])
    print(name)

    rows = []
    for fold_idx, (train_idx, val_idx) in enumerate(splits, 1):
        fold_dir = exp_dir / f"fold_{fold_idx}"
        if not (fold_dir / "best_weights.index").exists():
            print(f"  fold {fold_idx}: sin pesos, se salta")
            continue

        df_vl    = df_trainval.iloc[val_idx].reset_index(drop=True)
        y_vl_int = np.argmax(df_vl[class_cols].values.astype("float32"), axis=1)
        val_ds   = make_ds(df_vl["filepath"].values, cfg["img_size"], cfg["preprocess"])

        tf.keras.backend.clear_session()
        built = cfg["build_fn"](cfg["img_size"])
        model = built[0] if isinstance(built, tuple) else built
        for xb in val_ds.take(1):
            _ = model(xb[:1], training=False)
        model.load_weights(str(fold_dir / "best_weights")).expect_partial()

        proba = model.predict(val_ds, verbose=0)
        if isinstance(proba, (list, tuple)):   # U-Net autoencoder devuelve [recon, clasif]
            proba = proba[1]

        m = compute_all_metrics(y_vl_int, proba)
        rows.append({"fold": fold_idx, **m})
        print(f"  fold {fold_idx}: f1_macro={m['disease_f1_macro']:.4f}  "
              f"bal_acc={m['disease_balanced_acc']:.4f}")

    df_val = pd.DataFrame(rows)
    df_val.to_csv(exp_dir / "all_folds_VAL_metrics.csv", index=False)

    num  = [c for c in df_val.columns if c != "fold"]
    summ = df_val[num].agg(["mean", "median", "std"]).T
    summ.columns = ["mean", "median", "std"]
    summ.to_csv(exp_dir / "summary_VAL_mean_median_std.csv")

    print(f"  rendimiento esperado (validacion) — {name}:")
    print(summ[["mean", "std"]].round(4))
    print(f"  guardado en: {exp_dir}")
    print()

VGG16


2026-06-23 09:53:55.427313: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcuda.so.1
2026-06-23 09:53:55.473186: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1716] Found device 0 with properties: 
pciBusID: 0000:06:00.0 name: GeForce GTX 1080 Ti computeCapability: 6.1
coreClock: 1.582GHz coreCount: 28 deviceMemorySize: 10.92GiB deviceMemoryBandwidth: 451.17GiB/s
2026-06-23 09:53:55.473241: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1
2026-06-23 09:53:55.476155: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcublas.so.10
2026-06-23 09:53:55.478676: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcufft.so.10
2026-06-23 09:53:55.479099: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcurand.so

  fold 1: f1_macro=0.7019  bal_acc=0.7142
  fold 2: f1_macro=0.6776  bal_acc=0.6961
  fold 3: f1_macro=0.6880  bal_acc=0.7135
  fold 4: f1_macro=0.6775  bal_acc=0.7215
  fold 5: f1_macro=0.7162  bal_acc=0.7406
  rendimiento esperado (validacion) — VGG16:
                           mean     std
disease_acc              0.7892  0.0186
disease_balanced_acc     0.7172  0.0161
disease_precision_macro  0.6791  0.0204
disease_recall_macro     0.7172  0.0161
disease_f1_macro         0.6922  0.0167
disease_f1_weighted      0.7994  0.0149
disease_kappa            0.6243  0.0246
disease_auc_macro        0.9529  0.0068
auc_melanoma             0.8950  0.0154
  guardado en: /home/marc/MARIADELMAR_EXPERIMENTS/STL_Disease_experimentos/VGG16_STL/exp_2026-04-28_18-45_disease_CUT7_5fold

ResNet50
  fold 1: f1_macro=0.6632  bal_acc=0.7105
  fold 2: f1_macro=0.7352  bal_acc=0.7748
  fold 3: f1_macro=0.6911  bal_acc=0.7385
  fold 4: f1_macro=0.7070  bal_acc=0.7205
  fold 5: f1_macro=0.7236  bal_acc=0.7563


In [6]:
for cfg in MODELS:
    exp_dir = Path(cfg["exp_dir"])
    val_csv_f  = exp_dir / "all_folds_VAL_metrics.csv"
    test_csv_f = exp_dir / "all_folds_metrics.csv"
    if not (val_csv_f.exists() and test_csv_f.exists()):
        print(f"{cfg['name']}: falta algun CSV, se salta"); continue
    f1_val  = pd.read_csv(val_csv_f)["disease_f1_macro"].mean()
    f1_test = pd.read_csv(test_csv_f)["disease_f1_macro"].mean()
    print(f"{cfg['name']:<14} f1_val={f1_val:.4f}  f1_test={f1_test:.4f}  "
          f"brecha={f1_val - f1_test:+.4f}")

VGG16          f1_val=0.6922  f1_test=0.6636  brecha=+0.0287
ResNet50       f1_val=0.7040  f1_test=0.6923  brecha=+0.0117
MobileNetV2    f1_val=0.6637  f1_test=0.6211  brecha=+0.0426
DenseNet169    f1_val=0.7224  f1_test=0.7033  brecha=+0.0191
UNet           f1_val=0.4556  f1_test=0.4462  brecha=+0.0094


In [7]:
import pandas as pd
from pathlib import Path

METRICS_TO_CHECK = [
    "disease_f1_macro", "disease_balanced_acc",
    "disease_auc_macro", "auc_melanoma",
]

rows = []
for cfg in MODELS:
    exp_dir = Path(cfg["exp_dir"])
    val_csv_f  = exp_dir / "all_folds_VAL_metrics.csv"
    test_csv_f = exp_dir / "all_folds_metrics.csv"
    if not (val_csv_f.exists() and test_csv_f.exists()):
        print(f"{cfg['name']}: falta algun CSV, se salta"); continue
    dv = pd.read_csv(val_csv_f)
    dt = pd.read_csv(test_csv_f)
    for m in METRICS_TO_CHECK:
        if m not in dv.columns or m not in dt.columns: 
            continue
        rows.append({
            "modelo": cfg["name"], "metrica": m,
            "val_mean": dv[m].mean(), "val_std": dv[m].std(),
            "test_mean": dt[m].mean(), "test_std": dt[m].std(),
            "brecha_val_test": dv[m].mean() - dt[m].mean(),
            "n_folds_val": len(dv), "n_folds_test": len(dt),
        })
print(pd.DataFrame(rows).round(4).to_string(index=False))

     modelo              metrica  val_mean  val_std  test_mean  test_std  brecha_val_test  n_folds_val  n_folds_test
      VGG16     disease_f1_macro    0.6922   0.0167     0.6636    0.0164           0.0287            5             5
      VGG16 disease_balanced_acc    0.7172   0.0161     0.6798    0.0205           0.0373            5             5
      VGG16    disease_auc_macro    0.9529   0.0068     0.9494    0.0045           0.0034            5             5
      VGG16         auc_melanoma    0.8950   0.0154     0.8916    0.0099           0.0034            5             5
   ResNet50     disease_f1_macro    0.7040   0.0283     0.6923    0.0264           0.0117            5             5
   ResNet50 disease_balanced_acc    0.7401   0.0261     0.7187    0.0230           0.0214            5             5
   ResNet50    disease_auc_macro    0.9655   0.0035     0.9616    0.0029           0.0039            5             5
   ResNet50         auc_melanoma    0.9154   0.0090     0.9188  